<a href="https://colab.research.google.com/github/Jaya9522/almabetter_assignments/blob/main/M1W3_Guided_Project_EDA_on_IMDB_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## You have been hired by a rookie movie producer to help him decide what type of movies to produce and which actors to cast. You have to back your recommendations based on thorough analysis of the data he shared with you which has the list of 3000 movies and the corresponding details.

## As a data scientist, you have to first explore the data and check its sanity.

## Further, you have to answer the following questions:
1. ### <b> Which movie made the highest profit? Who were its producer and director? Identify the actors in that film.</b>
2. ### <b>This data has information about movies made in different languages. Which language has the highest average ROI (return on investment)? </b>
3. ### <b> Find out the unique genres of movies in this dataset.</b>
4. ### <b> Make a table of all the producers and directors of each movie. Find the top 3 producers who have produced movies with the highest average RoI? </b>
5. ### <b> Which actor has acted in the most number of movies? Deep dive into the movies, genres and profits corresponding to this actor. </b>
6. ### <b>Top 3 directors prefer which actors the most? </b>



# Data Exploration

In [ ]:
#Import package
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path='/content/drive/MyDrive/DATA SET/'
imdb_df=  pd.read_csv(file_path + 'imdb_data_set.csv')

In [ ]:
imdb_df.head()

In [ ]:
imdb_df_new = imdb_df.copy()

In [ ]:
#converting string of list into list
def str_to_list(string):
  return eval(string)

In [ ]:
imdb_df_new['genres'] = imdb_df_new.loc[~imdb_df_new['genres'].isna(),'genres'].apply(str_to_list)

In [ ]:
imdb_df_new['cast'] = imdb_df_new.loc[~imdb_df_new['cast'].isna(),'cast'].apply(str_to_list)

In [ ]:
imdb_df_new['crew'] = imdb_df_new.loc[~imdb_df_new['crew'].isna(),'crew'].apply(str_to_list)

In [ ]:
def movie_type(lst):
  type_of_movie = []
  try:
    for movie_types in lst:
      type_of_movie.append(movie_types['name'])
  except:
      return None

  return type_of_movie 

In [ ]:
#imdb_df_new[imdb_df_new['genres'][0][0]['name']]
imdb_df_new['movie'] = imdb_df_new['genres'].apply(movie_type)

In [ ]:
def prod_direc(crew_list):
  try:
    prod_list = []
    direc_list = []

    for member in crew_list:
      if member['job'] == 'Producer':
        prod_list.append(member['name'])
      elif member['job'] == 'Director':
        direc_list.append(member['name'])
    return prod_list,direc_list
  except:
    return [],[]      

In [ ]:
imdb_df_new[['producers','directors']] = imdb_df_new.apply(lambda x : prod_direc(x['crew']),axis=1,result_type='expand')

In [ ]:
def actors_list(cast_list):
  try:
    actor_list = []
    for member in cast_list:
      actor_list.append(member['name'])
    return actor_list
  except:
    return [] 

In [ ]:
imdb_df_new['actors'] = imdb_df_new['cast'].apply(actors_list)

In [ ]:
#Extremely low budget movie and revenue replace by median
imdb_df_new.loc[imdb_df_new['budget'] < 1000, 'budget'] = imdb_df_new.loc[imdb_df_new['budget'] >= 1000, 'budget'].median()
imdb_df_new.loc[imdb_df_new['revenue'] < 1000, 'revenue'] = imdb_df_new.loc[imdb_df_new['revenue'] >= 1000, 'revenue'].median()

In [ ]:
imdb_df_new['profit'] = imdb_df_new['revenue'] - imdb_df_new['budget']

In [ ]:
imdb_df_new.head()

In [ ]:
#max_profit
imdb_df_new['profit'].max()

In [ ]:
imdb_df_new.loc[imdb_df_new['profit'] == 1316249360, ['title','producers','directors','actors']]

In [ ]:
#.idxmax()-->> returns the row number(index) for the max value of the column 
max_movie_id = imdb_df_new.iloc[imdb_df_new['profit'].idxmax()]

In [ ]:
#movie_name
max_movie_id['title']

In [ ]:
max_movie_id['producers']

In [ ]:
max_movie_id['directors']

In [ ]:
#movie_actors
print(max_movie_id['actors'])

**This data has information about movies made in different languages. Which language has the highest average ROI (return on investment)?**

In [ ]:
#calculate ROI
imdb_df_new['roi'] = 100 * imdb_df_new['profit']/imdb_df_new['budget']

In [ ]:
imdb_df_new.groupby('original_language')['roi'].mean().sort_values(ascending=False).reset_index()[:3]

**Find out the unique genres of movies in this dataset.**

In [ ]:
genres = imdb_df_new['movie'] 

In [ ]:
genres = genres.replace('None',np.nan).dropna()

In [ ]:
print(list(set(genres.explode('movie'))))

**Make a table of all the producers and directors of each movie. Find the top 3 producers who have produced movies with the highest average RoI?**

In [ ]:
# Your code here
movie_directors_producers = {'Movie': imdb_df_new['original_title'], 'Producers': imdb_df_new['producers'], 'Directors': imdb_df_new['directors']}

In [ ]:
df1 = pd.DataFrame(movie_directors_producers)
df1[:5]

In [ ]:
movie_producers_avg_roi = {'Producers': imdb_df_new['producers'],'Movie': imdb_df_new['original_title'], 'ROI': imdb_df_new['roi']}

In [ ]:
test_df = pd.DataFrame(movie_producers_avg_roi)

In [ ]:
test_df = test_df.explode('Producers',ignore_index=True)

In [ ]:
prod_list = list(test_df['Producers'])

In [ ]:
unique_prod_list = []
for x in prod_list:
  if x not in unique_prod_list:
    unique_prod_list.append(x)

In [ ]:
movi_list = list(test_df['Movie'])

In [ ]:
roi_list = list(test_df['ROI'])

In [ ]:
movie_lst = []                                                #list of list(movies)
avg_roi_lst = []

def find_avg_roi(unique_prod_list):
  for prod_name in unique_prod_list:
    movie_name = []
    avg_roi = []
    for i in range(len(prod_list)):
      if prod_list[i] == prod_name:
        movie_name.append(movi_list[i])
        avg_roi.append(roi_list[i])

    movie_lst.append(movie_name)                               #list of movie 
    try:
      avg_roi_lst.append(sum(avg_roi)/len(avg_roi))            #avg of ROI
    except:
      avg_roi_lst.append(np.nan)                               #to avoid ZeroDivision Error
         

In [ ]:
find_avg_roi(unique_prod_list)

In [ ]:
avg_roi = {'Producer': unique_prod_list, 'Movies' : movie_lst, 'Average_ROI':avg_roi_lst }

In [ ]:
df3 = pd.DataFrame(avg_roi)

In [ ]:
df3.sort_values(by='Average_ROI', ascending=False)[:3]

**Which actor has acted in the most number of movies? Deep dive into the movies, genres and profits corresponding to this actor.**

In [ ]:
max_movies = {'movie_title':imdb_df_new['original_title'], 'movie_genres': imdb_df_new['movie'], 'actors': imdb_df_new['actors'],'profit': imdb_df_new['profit']}

In [ ]:
df_actor_movie = pd.DataFrame(max_movies)

In [ ]:
#The explode() function is used to transform each element of a list-like to a row
df_actor_movie = df_actor_movie.explode('actors',ignore_index=True)

In [ ]:
#actor who acted most no.of movies
df_actor_movie.groupby(['actors'])['movie_title'].count().sort_values(ascending=False).reset_index()[:5]

In [ ]:
#final_data_frame
df_actor_movie[df_actor_movie['actors'] == 'Samuel L. Jackson'].sort_values(by='profit', ascending=False)[:5]

**Top 3 directors prefer which actors the most?**

In [ ]:
# Your code here 
director_actor = {'directors': imdb_df_new['directors'], 'actors': imdb_df_new['actors']}

In [ ]:
top_directors = pd.DataFrame(director_actor)
top_directors[:5]

In [ ]:
#The explode() function is used to transform each element of a list-like to a row
top_directors = top_directors.explode('actors',ignore_index=True)

In [ ]:
#The explode() function is used to transform each element of a list-like to a row
top_directors = top_directors.explode('directors',ignore_index=True)

In [ ]:
#top three directors
top_directors.groupby(['directors'])['actors'].count().sort_values(ascending=False).reset_index()[:3]

In [ ]:
#top-1
top_directors[top_directors['directors'] == 'Ron Howard'][:5]

In [ ]:
#top-2
top_directors[top_directors['directors'] == 'Steven Spielberg'][:5]

In [ ]:
#top-3
top_directors[top_directors['directors'] == 'Todd Phillips'][:5]